In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

In [ ]:
results_dir = Path.cwd() / "Results"

summary_files = {
    "NC": results_dir / "NC_SDDP.csv",
    "OH": results_dir / "OH_SDDP.csv",
}


def load_master_summary(dataset="NC"):
    dataset = dataset.upper()

    if dataset not in summary_files:
        raise KeyError(
            f"Unknown dataset {dataset!r}; expected one of {list(summary_files)}"
        )

    path = summary_files[dataset]

    if not path.exists():
        raise FileNotFoundError(f"Missing summary file: {path}")

    return pd.read_csv(path)


def parse_selected_gmm_k(value):
    if pd.isna(value):
        return np.nan

    vals = []
    for p in str(value).split(","):
        p = p.strip()
        if p and p.lower() not in {"nan", "none"}:
            vals.append(float(p))

    return np.mean(vals) if vals else np.nan


def summarize(dataset):
    dataset = dataset.upper()
    df = load_master_summary(dataset)

    required = ["method", "OOS_mean", "p10", "p90"]

    missing = [col for col in required if col not in df.columns]
    if missing:
        raise KeyError(
            f"{dataset} summary is missing expected columns {missing}. "
            f"Available columns: {list(df.columns)}"
        )

    for col in ["OOS_mean", "p10", "p90"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    if "selected_gmm_K" in df.columns:
        df["selected_gmm_K_mean"] = df["selected_gmm_K"].apply(parse_selected_gmm_k)

    mean_cols = [
        "selected_gmm_K_mean",
        "best_ETA_raw",
        "best_eta_scaled",
        "NUM_NEURAL_CUTS",
        "NEURAL_REFINEMENT_ITER",
    ]

    time_cols = [
        "runtime_total_seconds",
        "time_transition_seconds",
        "time_eta_tuning_seconds",
        "time_sddp_solve_seconds",
        "time_VAL_eval_seconds",
        "time_validation_eval_seconds",
        "time_test_eval_seconds",
        "time_nn_teacher_sddp_seconds",
        "time_nn_training_seconds",
        "time_nn_prediction_seconds",
        "time_nn_final_forward_seconds",
        "time_nn_total_with_teacher_seconds",
        "time_nn_inference_excluding_teacher_seconds",
    ]

    optional_aggs = {}

    for col in mean_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            optional_aggs[col] = (col, "mean")

    for col in time_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            optional_aggs[col] = (col, "sum")

    summary = (
        df.groupby("method", dropna=False)
        .agg(
            mean=("OOS_mean", "mean"),
            p10=("p10", "mean"),
            p90=("p90", "mean"),
            num_quarters=("method", "size"),
            **optional_aggs,
        )
        .reindex(["IND", "NW", "GMM", "NN"])
    )

    return summary.round(3).add_prefix(f"{dataset}_")


def try_summarize(dataset):
    try:
        return summarize(dataset)
    except FileNotFoundError as exc:
        print(exc)
        return None


compare_NC = try_summarize("NC")
compare_OH = try_summarize("OH")

print("NC summary")
print(compare_NC)

print("OH summary")
print(compare_OH)